# Aula 2 — Laboratório: Métricas de classificação I

**Como usar este caderno.** Cada parte traz um **exemplo já resolvido** (código pronto, que você roda e lê) e, logo depois, um **exercício de aplicação**: você escreve um trecho parecido, agora sobre outra coluna, outra métrica ou outro limiar. Rode as células na ordem.

Contexto: triagem de **risco metabólico**. O alvo é `alto_risco` (positivo = paciente de **alto** risco), uma classe **rara** (~15%).

## Parte 0 — Ambiente e dados

Tudo aqui já está pronto: importe, carregue, monte o alvo binário e treine um classificador. É a base para todas as partes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report)

pac = pd.read_csv("datasets/1_pacientes.csv")
pac["alto_risco"] = (pac["risco_metabolico"] == "alto").astype(int)   # alvo binário

# features: tudo menos identificador e a coluna de origem do alvo; sexo vira dummy
X = pd.get_dummies(pac.drop(columns=["id","risco_metabolico","alto_risco"]), columns=["sexo"], drop_first=True)
y = pac["alto_risco"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)   # stratify mantém os ~15%

modelo = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)
print("Treino:", X_tr.shape[0], "| Teste:", X_te.shape[0])
print("Alto risco no teste:", int(y_te.sum()), "de", len(y_te))

## Parte 1 — A base: a proporção da classe

Antes de qualquer métrica, precisamos saber **quão rara** é a classe positiva. Se o alvo é raro, a acurácia vai enganar (Parte 6).

**Exemplo resolvido** — proporção de alto risco no conjunto **completo**:

In [ ]:
prop_total = y.mean()
print(f"Alto risco no total: {prop_total:.3f}  ({prop_total*100:.1f}%)")

### Exercício 1 — aplicação
Faça o mesmo para **treino** (`y_tr`) e **teste** (`y_te`). Eles devem ficar próximos de ~15% — é o que o `stratify` garante.

In [ ]:
# Aplique o modelo do exemplo a y_tr e y_te
prop_tr = ...
prop_te = ...
print(f"treino: {prop_tr:.3f} | teste: {prop_te:.3f}")

*Sua resposta (1 frase):* por que treino e teste têm proporções parecidas?

## Parte 2 — A matriz de confusão

As quatro caixas: **VN, FP, FN, VP**. Lembre: positivo = doente (alto risco).

**Exemplo resolvido** — matriz e as quatro caixas:

In [ ]:
cm = confusion_matrix(y_te, y_pred)      # ordem das classes: 0 (baixo), 1 (alto)
vn, fp, fn, vp = cm.ravel()              # ravel segue VN, FP, FN, VP
print(cm)
print(f"VN={vn}  FP={fp}  FN={fn}  VP={vp}")

fig, ax = plt.subplots(figsize=(4.2,3.4))
ConfusionMatrixDisplay(cm, display_labels=["baixo","alto"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusão — alto risco"); plt.show()

### Exercício 2 — aplicação
Sem recalcular a matriz, responda **com as variáveis** `vn, fp, fn, vp`:
- (a) quantos pacientes de **alto risco o modelo deixou passar** (classificou como baixo)?
- (b) quantos **alarmes falsos** (baixo risco marcados como alto)?

*Dica:* um é o **FN**, o outro é o **FP**.

In [ ]:
# (a) doentes que passaram = ...   (b) alarmes falsos = ...
passaram   = ...
falsos_alarmes = ...
print("Alto risco que passou (FN):", passaram)
print("Alarmes falsos (FP):", falsos_alarmes)

*Sua resposta:* numa triagem de doença, qual dos dois erros é o mais grave — e por quê?

## Parte 3 — Acurácia

Fração de acertos sobre o total. Simples — e traiçoeira quando a classe é rara.

**Exemplo resolvido** — pela função e 'na mão':

In [ ]:
acc = accuracy_score(y_te, y_pred)
acc_manual = (vp + vn) / (vp + vn + fp + fn)
print(f"acurácia (função): {acc:.3f}")
print(f"acurácia (na mão): {acc_manual:.3f}")

### Exercício 3 — aplicação
Confirme que a acurácia é a **taxa de acerto**: calcule `(vp+vn)` e divida pelo total. Deve bater com o exemplo.

In [ ]:
total = ...
acertos = ...
print("acertos/total =", round(acertos/total, 3))

## Parte 4 — Precisão e recall

Duas perguntas diferentes:
- **Precisão** = VP / (VP+FP) — *dos que sinalizei, quantos eram mesmo?* (pune o **FP**)
- **Recall** = VP / (VP+FN) — *dos que eram, quantos peguei?* (pune o **FN**)

**Exemplo resolvido** — pela função:

In [ ]:
prec = precision_score(y_te, y_pred)
rec  = recall_score(y_te, y_pred)
print(f"precisão: {prec:.3f}")
print(f"recall  : {rec:.3f}")

### Exercício 4 — aplicação
Calcule **precisão e recall 'na mão'** a partir de `vp, fp, fn` e confira com o exemplo.
*(É exatamente a Questão 2 do formulário: precisão = VP/(VP+FP).)*

In [ ]:
prec_manual = ...
rec_manual  = ...
print(f"precisão (na mão): {prec_manual:.3f}")
print(f"recall   (na mão): {rec_manual:.3f}")

*Sua resposta:* qual afirmação está certa — 'precisão pune o FP; recall pune o FN' ou o contrário?

## Parte 5 — F1 e a média harmônica

O **F1** combina precisão e recall pela **média harmônica** — que **puxa para o menor** dos dois. Um número alto não mascara um baixo.

**Exemplo resolvido** — F1 pela função e o contraste harmônica × aritmética num caso extremo:

In [ ]:
f1 = f1_score(y_te, y_pred)
print(f"F1 do modelo: {f1:.3f}")

# caso extremo: precisão alta, recall baixo
P, R = 0.9, 0.1
aritmetica = (P + R) / 2
harmonica  = 2 * P * R / (P + R)
print(f"P=0.9, R=0.1  ->  aritmética={aritmetica:.2f}  |  harmônica (F1)={harmonica:.2f}")

### Exercício 5 — aplicação
Calcule o **F1 'na mão'** a partir de `prec` e `rec` (média harmônica) e confira com `f1`.
Depois responda: por que a harmônica é preferível à aritmética aqui?
*(É a Questão 5 do formulário.)*

In [ ]:
f1_manual = ...   # 2*P*R/(P+R) com P=prec, R=rec
print(f"F1 (na mão): {f1_manual:.3f}")

*Sua resposta:* em uma frase, por que o F1 usa a média harmônica e não a aritmética?

## Parte 6 — A armadilha da acurácia (o *dummy*)

Um modelo que **sempre prevê a classe majoritária** (baixo risco) acerta ~85% — e não detecta **nenhum** doente.

**Exemplo resolvido** — o dummy:

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
y_dummy = dummy.predict(X_te)
print("acurácia do DUMMY:", round(accuracy_score(y_te, y_dummy), 3))
print("recall  do DUMMY:", round(recall_score(y_te, y_dummy, zero_division=0), 3))

### Exercício 6 — aplicação
Monte uma pequena **tabela comparando** o dummy e a árvore em acurácia, precisão, recall e F1.
Qual métrica **revela** que o dummy é inútil? *(Questão 4 do formulário.)*

In [ ]:
def resumo(nome, yv, yp):
    return {'modelo':nome,
            'acuracia':round(accuracy_score(yv,yp),3),
            'precisao':round(precision_score(yv,yp,zero_division=0),3),
            'recall':round(recall_score(yv,yp,zero_division=0),3),
            'f1':round(f1_score(yv,yp,zero_division=0),3)}
# TODO: crie o DataFrame com as duas linhas (dummy e árvore)
tabela = pd.DataFrame([...])
tabela

*Sua resposta:* a acurácia do dummy é alta; qual métrica denuncia que ele não serve?

## Parte 7 — O trade-off do limiar

O modelo devolve uma **probabilidade**; o **limiar** a transforma em decisão. **Baixar** o limiar marca mais gente como positiva: **recall sobe, precisão cai**.

**Exemplo resolvido** — varrendo três limiares:

In [ ]:
proba = modelo.predict_proba(X_te)[:, 1]      # probabilidade de alto risco
for limiar in [0.50, 0.30, 0.20]:
    pred_l = (proba >= limiar).astype(int)
    r = recall_score(y_te, pred_l, zero_division=0)
    p = precision_score(y_te, pred_l, zero_division=1)
    print(f"limiar {limiar:.2f} | recall={r:.2f} | precisão={p:.2f}")

### Exercício 7 — aplicação
Escolha **um limiar mais alto** (ex.: 0.70) e mostre o efeito **oposto**: precisão sobe, recall cai. Imprima recall e precisão nesse limiar.

In [ ]:
limiar_alto = 0.70
pred_alto = ...
# imprima recall e precisão em pred_alto

### Exercício 8 — aplicação (gráfico)
Faça o gráfico do **trade-off**: para vários limiares, plote **precisão** e **recall** na mesma figura.
*Dica:* percorra `np.linspace(0.05, 0.9, 30)` e guarde as duas listas.

In [ ]:
limiares = np.linspace(0.05, 0.9, 30)
precs, recs = [], []
for L in limiares:
    pred_L = ...
    precs.append(...)
    recs.append(...)
# plt.plot(limiares, recs, label='recall'); plt.plot(limiares, precs, label='precisão')
# plt.xlabel('limiar'); plt.legend(); plt.show()

*Sua resposta:* descreva com suas palavras o que o gráfico mostra ao **baixar** o limiar. *(Questão 6 do formulário.)*

## Parte 8 — Escolher a métrica pelo custo do erro

A métrica a priorizar depende de **qual erro dói mais**. Numa triagem de doença grave, o pior erro é o **FN** (doente não detectado) — priorize **recall**.

**Exemplo resolvido** — priorizando recall com um limiar mais baixo:

In [ ]:
limiar_triagem = 0.20   # baixamos para não perder doentes
pred_tri = (proba >= limiar_triagem).astype(int)
print("Cenário TRIAGEM (recall prioritário):")
print(f"  recall={recall_score(y_te, pred_tri, zero_division=0):.2f} | "
      f"precisão={precision_score(y_te, pred_tri, zero_division=1):.2f}")

### Exercício 9 — aplicação
Cenário diferente: uma **campanha de marketing** cara, em que cada contato custa. Aqui o pior erro é o **FP** (gastar com quem não interessa) — priorize **precisão**.
Escolha um limiar **mais alto** para esse objetivo e imprima recall/precisão. *(Questão 7 trata do caso oposto.)*

In [ ]:
limiar_mkt = ...   # alto, para priorizar precisão
pred_mkt = ...
# imprima recall e precisão

*Sua resposta:* explique por que triagem prioriza recall e marketing prioriza precisão.

## Parte 9 — O relatório e as médias (macro × weighted)

O `classification_report` imprime precisão, recall e F1 de cada classe, mais duas médias:
- **macro**: média simples entre as classes (cada classe pesa igual → dá voz à rara)
- **weighted**: média ponderada pelo *support* (a classe grande pesa mais)

**Exemplo resolvido** — o relatório completo:

In [ ]:
print(classification_report(y_te, y_pred, target_names=["baixo","alto"]))

### Exercício 10 — aplicação
Extraia o relatório como dicionário (`output_dict=True`) e imprima **só** o F1 macro e o F1 weighted. Qual dos dois é mais afetado pela classe rara 'alto'? *(Questão 8 do formulário.)*

In [ ]:
rep = classification_report(y_te, y_pred, target_names=['baixo','alto'], output_dict=True)
f1_macro    = ...
f1_weighted = ...
print(f"F1 macro={f1_macro:.3f} | F1 weighted={f1_weighted:.3f}")

*Sua resposta:* por que a média **macro** costuma ser mais baixa que a **weighted** aqui?

## Parte 10 — Síntese

Responda em poucas linhas, usando o que você calculou:
1. Neste problema (doença rara), qual métrica você reportaria como principal e por quê?
2. Que limiar você usaria e qual o custo dessa escolha?
3. Por que a acurácia sozinha seria uma má escolha aqui?

*Suas conclusões:*

---
*Fim do laboratório.*